# 11. Principal Component Analysis (PCA)
**Use case:** Financial Services — Treasury yield curve risk factors for rate hedging

A bank's treasury and rates desk holds bonds and swaps across maturities from 3 months to 30 years, and every daily rate move changes their value. Hedging each maturity separately is costly and noisy, and risk managers need a simple story for why the portfolio gained or lost. Daily changes in Treasury yields are freely available and highly correlated, which makes them ideal for PCA. Success means three named factors (level, slope, curvature) that explain about 99% of curve moves, a factor-based VaR, and a small set of hedge trades that cut daily P&L swings sharply.

> 💡 Imagine 10 friends on a seesaw-shaped bench: when one end goes up, everyone near it goes up too. Instead of tracking all 10 heads, you track just 'how high is the whole bench' and 'how tilted is it' - two numbers that tell you almost everything.

Legend in code: `# 🐍 PY:` Python lesson · `# 🧠 ML:` ML lesson

In [ ]:
# Run once (Colab already has most of these)
!pip -q install numpy pandas scikit-learn scipy xgboost torch

## EASY: Find level, slope and curve by hand on 10 days
Compute PCA from scratch with numpy on 10 days x 4 maturities and see the level factor emerge.

| 5W + H | |
|---|---|
| **Who** | A junior risk analyst learning why the desk talks about 'level' and 'slope'. |
| **What** | Ten typed-in days of yield changes for 3M, 2Y, 10Y and 30Y, decomposed into factors. |
| **When** | Before trusting a library, to understand what PCA actually computes. |
| **Where** | A notebook on a laptop - no market data feed needed. |
| **Why** | Seeing the covariance matrix and eigenvectors makes later library output easy to read. |
| **How** | Centre the columns, build the covariance matrix, call np.linalg.eigh, sort, and project. |

**🧠 ML concepts:** Centring data before PCA, Covariance matrix, Eigenvectors as factor shapes, Eigenvalues as explained variance, Projection to factor scores  
**🐍 Python concepts:** Lists and 2-D numpy arrays, axis=0 aggregation, Broadcasting subtraction, Slicing with [::-1] to reverse, enumerate and f-string formatting, Matrix multiply with @

In [ ]:
import numpy as np

# Daily changes in Treasury yields, in basis points (1 bp = 0.01%), for 10 trading days
# 🐍 PY: a list of strings - we use it later only for printing nice column headers
maturities = ["3M", "2Y", "10Y", "30Y"]
# 🐍 PY: np.array turns a list of lists into a 2-D table: 10 rows (days) x 4 columns (maturities)
changes_bp = np.array([
    [ 2.0,  4.0,  5.0,  5.0],
    [-1.0, -3.0, -4.0, -4.0],
    [ 0.5,  2.0,  1.0,  0.0],
    [ 3.0,  6.0,  7.0,  6.0],
    [-2.0, -4.0, -6.0, -6.0],
    [ 1.0,  0.0, -2.0, -3.0],
    [ 0.0,  1.0,  3.0,  4.0],
    [-1.5, -5.0, -5.0, -4.0],
    [ 2.5,  3.0,  4.0,  3.0],
    [-0.5,  1.0,  0.0, -1.0],
])
print("Data shape (days, maturities):", changes_bp.shape)

# 🧠 ML: PCA works on deviations from the average, so subtract each column's mean first
# 🐍 PY: axis=0 means 'down the rows', giving one mean per column; broadcasting subtracts it from every row
centered = changes_bp - changes_bp.mean(axis=0)

# 🧠 ML: the covariance matrix says how each pair of maturities moves together
cov = np.cov(centered, rowvar=False)
print("\nCovariance matrix (bp^2):")
print(np.round(cov, 1))

# 🧠 ML: eigenvectors = directions of movement; eigenvalues = how much variance each direction carries
eigvals, eigvecs = np.linalg.eigh(cov)
# 🐍 PY: eigh returns smallest first; [::-1] reverses the order so the biggest factor comes first
order = np.argsort(eigvals)[::-1]
eigvals, eigvecs = eigvals[order], eigvecs[:, order]
# 🧠 ML: an eigenvector's sign is arbitrary; flip so the 30Y loading is positive (easier to read)
eigvecs = eigvecs * np.sign(eigvecs[-1, :])

share = eigvals / eigvals.sum()
names = ["PC1 level?", "PC2 slope?", "PC3 curve?", "PC4 noise"]
print("\nFactor       variance share   loadings on", maturities)
# 🐍 PY: enumerate gives the position i and the item together; f-strings format numbers neatly
for i, name in enumerate(names):
    print(f"{name:<12} {share[i]:>8.1%}        {np.round(eigvecs[:, i], 2)}")

# 🧠 ML: projecting each day onto PC1 gives one 'level move' number per day
pc1_scores = centered @ eigvecs[:, 0]
print("\nDay-by-day PC1 score (how much the whole curve shifted):")
print(np.round(pc1_scores, 1))

**Try this next:**
- Change day 6 so the short end rises and the long end falls sharply, and watch PC2's share grow.
- Remove the sign-flip line and rerun - the loadings may come out negative, with identical meaning.
- Add a fifth maturity column (5Y) and check that PC1 still dominates.

## MEDIUM: sklearn PCA on 6 years of synthetic yield moves
Fit sklearn PCA on 1,500 days of 8-maturity yield changes, read the factor shapes, and test reconstruction on later days.

| 5W + H | |
|---|---|
| **Who** | A market-risk analyst building the desk's factor model. |
| **What** | 1,500 daily changes across 8 Treasury maturities, reduced to 3 factors. |
| **When** | Monthly or quarterly when the factor model is refreshed. |
| **Where** | A Python batch job in the risk analytics environment. |
| **Why** | Three factors are easier to hedge, explain and report than eight correlated maturities. |
| **How** | Generate realistic data, split by time, fit PCA(n_components=3), inspect loadings and holdout error. |

**🧠 ML concepts:** Explained variance ratio, Loadings interpretation, Time-ordered train/test split, Reconstruction error as a quality metric, When not to standardise features  
**🐍 Python concepts:** Dictionaries and list(dict.values()), np.outer to build tables, List comprehension for column names, pandas DataFrame and insert, Boolean checks with .all()

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

np.random.seed(42)

# --- 1. Simulate 1,500 trading days of daily yield changes (bp) for 8 maturities ---
# 🐍 PY: a dict of maturity label -> years, used to build smooth factor shapes
tenors = {"3M": 0.25, "6M": 0.5, "1Y": 1, "2Y": 2, "5Y": 5, "10Y": 10, "20Y": 20, "30Y": 30}
years = np.array(list(tenors.values()))
n_days = 1500

# 🧠 ML: real curves move as a few hidden drivers - we bake in level, slope and curvature shapes
level_shape = np.ones(len(years))
slope_shape = np.log(years) / np.log(30)
curve_shape = np.exp(-((np.log(years) - np.log(3)) ** 2))
# 🐍 PY: np.random.normal(mean, std, size) draws daily shocks for each hidden driver
level = np.random.normal(0, 6.0, n_days)
slope = np.random.normal(0, 2.5, n_days)
curve = np.random.normal(0, 1.2, n_days)
noise = np.random.normal(0, 0.6, (n_days, len(years)))
# 🐍 PY: np.outer builds a (days x maturities) table: each day's shock times the factor shape
moves = (np.outer(level, level_shape) + np.outer(slope, slope_shape)
         + np.outer(curve, curve_shape) + noise)

# 🐍 PY: a DataFrame gives named columns like chg_10Y_bp instead of raw column numbers
cols = [f"chg_{t}_bp" for t in tenors]
df = pd.DataFrame(moves, columns=cols)
df.insert(0, "trade_date", pd.bdate_range("2019-01-02", periods=n_days))
print("Sample rows:")
print(df.head(3).to_string(index=False, float_format="%.2f"))

# --- 2. Split: fit factors on earlier days, check on later days (no shuffling for time data) ---
# 🧠 ML: shuffle=False keeps time order so we test on 'future' days the model never saw
train, test = train_test_split(df[cols], test_size=0.25, shuffle=False)
print(f"\nTrain days: {len(train)}, Test days: {len(test)}")

# --- 3. Fit PCA ---
# 🧠 ML: no scaling here on purpose - all columns are in the same unit (bp), so size matters
pca = PCA(n_components=3)
pca.fit(train)
print("\nExplained variance ratio:", np.round(pca.explained_variance_ratio_, 4))
print(f"Top 3 factors together explain {pca.explained_variance_ratio_.sum():.2%} of daily moves")

# 🧠 ML: loadings (components_) show how each maturity reacts to one unit of each factor
loadings = pd.DataFrame(pca.components_.T, index=list(tenors), columns=["PC1", "PC2", "PC3"])
# 🐍 PY: multiply each column by the sign of its 30Y value so the shapes print the same way every run
loadings = loadings * np.sign(loadings.loc["30Y"])
print("\nLoadings (rows = maturities):")
print(loadings.round(3).to_string())

# --- 4. How well do 3 numbers rebuild the full 8-point curve on unseen days? ---
# 🧠 ML: transform -> 3 factor scores; inverse_transform -> back to 8 maturities
rebuilt = pca.inverse_transform(pca.transform(test))
err = test.values - rebuilt
rmse_bp = np.sqrt((err ** 2).mean())
print(f"\nTest reconstruction RMSE: {rmse_bp:.2f} bp (typical daily move ~{test.values.std():.1f} bp)")

# --- 5. Name the factors with simple rules ---
pc1, pc2, pc3 = loadings["PC1"], loadings["PC2"], loadings["PC3"]
# 🐍 PY: boolean expressions like (pc1 > 0).all() return True/False for every-element checks
print("\nFactor interpretation:")
print(f"  PC1 all same sign?       {(pc1 > 0).all()}  -> LEVEL (parallel shift)")
print(f"  PC2 short vs long sign:  {pc2['3M']:+.2f} vs {pc2['30Y']:+.2f}  -> SLOPE (steepen/flatten)")
print(f"  PC3 belly vs wings:      {pc3['2Y']:+.2f} vs {pc3['3M']:+.2f}/{pc3['30Y']:+.2f}  -> CURVATURE (butterfly)")

# --- 6. Business view ---
print("\nBusiness read-out for the rates desk:")
print(f"  Hedging the LEVEL factor alone covers {pca.explained_variance_ratio_[0]:.0%} of daily risk.")
print(f"  Adding SLOPE and CURVE hedges brings that to {pca.explained_variance_ratio_.sum():.0%}.")
print("  So 3 hedge trades can replace hedging 8 maturities separately.")

**Try this next:**
- Set n_components=1 and compare the test RMSE - how much does slope matter?
- Wrap PCA in a pipeline with StandardScaler and see how the loadings change.
- Double the curvature shock std (1.2 to 2.4) and watch PC3's variance share rise.

## HARD: Factor VaR, hedge and stability checks for a bond book
Choose the number of factors, compute PCA-based VaR for a bond portfolio, solve a 3-trade factor hedge, backtest it and check factor stability.

| 5W + H | |
|---|---|
| **Who** | The market-risk team and rates traders, with model validation reviewing the work. |
| **What** | A saved PCA pipeline that drives daily factor exposures, VaR and hedge recommendations for a 10-tenor curve. |
| **When** | Run daily for exposures and VaR; recalibrated when backtests or stability checks flag drift. |
| **Where** | Inside the bank's risk engine batch, with the model stored in a registry. |
| **Why** | Factor hedging removes most rate risk with few trades, and regulators expect documented, backtested risk models. |
| **How** | Pipeline of centring plus PCA, k chosen by variance target and holdout error, DV01 mapped to factors, a linear solve for hedge sizes, a holdout P&L backtest and rolling-window loading comparisons. |

**🧠 ML concepts:** Choosing k with a variance target and holdout error, Sign convention for comparable loadings, Factor-based parametric VaR, Hedging by neutralising factor exposures, Out-of-sample backtesting and regime shifts, Stability of components over rolling windows  
**🐍 Python concepts:** Functions with default arguments and docstrings, sklearn Pipeline and named_steps, pandas Series label indexing, zip over two lists, if __name__ == '__main__' guard, tempfile and joblib for saving

In [ ]:
"""PCA yield-curve risk factors: choose k, interpret, PCA-VaR, factor hedge, stability check."""
import os
import tempfile
import warnings

import joblib
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
np.random.seed(42)
TENORS = ["3M", "6M", "1Y", "2Y", "3Y", "5Y", "7Y", "10Y", "20Y", "30Y"]
YEARS = np.array([0.25, 0.5, 1, 2, 3, 5, 7, 10, 20, 30])


def simulate_curve_moves(n_days=2000):
    """Daily yield changes (bp) driven by level/slope/curve plus a calmer and a stressed regime."""
    shapes = np.vstack([np.ones(len(YEARS)),
                        np.log(YEARS) / np.log(30),
                        np.exp(-((np.log(YEARS) - np.log(3)) ** 2))])
    # 🧠 ML: volatility regimes make factor stability worth checking - real markets do this too
    vol_mult = np.where(np.arange(n_days) > 1400, 1.8, 1.0)[:, None]
    shocks = np.random.normal(0, [6.0, 2.5, 1.2], (n_days, 3)) * vol_mult
    noise = np.random.normal(0, 0.6, (n_days, len(YEARS)))
    dates = pd.bdate_range("2018-01-02", periods=n_days)
    return pd.DataFrame(shocks @ shapes + noise, index=dates, columns=TENORS)


def choose_k(train, test, target=0.99, max_k=6):
    """Pick the smallest k reaching target explained variance; show holdout reconstruction error."""
    rows = []
    for k in range(1, max_k + 1):
        # 🧠 ML: centre only (no std scaling) - all tenors share the bp unit, so big movers should dominate
        pipe = Pipeline([("center", StandardScaler(with_std=False)), ("pca", PCA(n_components=k))])
        pipe.fit(train)
        rebuilt = pipe.inverse_transform(pipe.transform(test))
        rmse = np.sqrt(((test.values - rebuilt) ** 2).mean())
        cum = pipe.named_steps["pca"].explained_variance_ratio_.sum()
        rows.append({"k": k, "cum_var": cum, "test_rmse_bp": rmse})
    table = pd.DataFrame(rows)
    k_star = int(table.loc[table.cum_var >= target, "k"].min())
    return table, k_star


def oriented_loadings(pca):
    """Return loadings with a fixed sign convention: positive at 30Y."""
    L = pd.DataFrame(pca.components_.T, index=TENORS,
                     columns=[f"PC{i + 1}" for i in range(pca.n_components_)])
    # 🧠 ML: eigenvector signs are arbitrary, so fix them before comparing windows or reporting
    return L * np.sign(L.loc["30Y"])


def pca_var(dv01, L, factor_var, z=2.326):
    """99% one-day parametric VaR using only the PCA factors. P&L = -DV01 . dy."""
    # 🧠 ML: exposure to each factor = sum over tenors of DV01 x loading
    expo = L.T.values @ dv01
    pnl_sd = np.sqrt(np.sum(expo ** 2 * factor_var))
    return z * pnl_sd, expo


def stability(df, window=250, step=250):
    """Cosine similarity of rolling-window loadings vs full-sample loadings."""
    full = oriented_loadings(PCA(3).fit(df - df.mean()))
    out = []
    for start in range(0, len(df) - window + 1, step):
        win = df.iloc[start:start + window]
        L = oriented_loadings(PCA(3).fit(win - win.mean()))
        # 🧠 ML: |cosine| near 1 means the factor shape is the same as the long-run shape
        cos = [abs(np.dot(L[c], full[c])) for c in full.columns]
        out.append({"window_start": win.index[0].date(), "PC1": cos[0], "PC2": cos[1],
                    "PC3": cos[2], "vol_bp": win.values.std()})
    return pd.DataFrame(out)


if __name__ == "__main__":
    df = simulate_curve_moves()
    # 🧠 ML: time-ordered split: fit on the past, evaluate on the most recent 400 days
    train, test = df.iloc[:-400], df.iloc[-400:]

    table, k = choose_k(train, test)
    print("Choosing number of factors (target 99% variance):")
    print(table.round(4).to_string(index=False))
    print(f"-> chosen k = {k}\n")

    pipe = Pipeline([("center", StandardScaler(with_std=False)), ("pca", PCA(n_components=k))])
    pipe.fit(train)
    pca = pipe.named_steps["pca"]
    L = oriented_loadings(pca)
    print("Loadings:")
    print(L.round(2).T.to_string())
    print("Interpretation: PC1=level (all +), PC2=slope (short - / long +), PC3=curvature (belly vs wings)\n")

    # Bond portfolio: key-rate DV01 in USD per 1bp RISE (long bonds lose money when yields rise)
    # 🧠 ML: DV01 is the 'feature weight' of each tenor in the P&L - risk = DV01 times rate move
    dv01 = pd.Series(0.0, index=TENORS)
    dv01[["2Y", "5Y", "10Y", "30Y"]] = [12_000, 25_000, 40_000, 18_000]
    factor_var = pca.explained_variance_
    var_before, expo = pca_var(dv01.values, L, factor_var)
    print("Portfolio factor exposures ($ per 1-unit factor move):", np.round(expo, 0))
    print(f"PCA 99% 1-day VaR before hedge: ${var_before:,.0f}")

    # 🧠 ML: factor hedge - pick hedge sizes so the portfolio has ZERO exposure to every factor
    hedge_tenors = ["2Y", "10Y", "30Y"][:k]
    H = L.loc[hedge_tenors].values.T  # (factors x hedge instruments), each hedge = 1 USD DV01
    hedge_dv01 = np.linalg.solve(H, -expo)
    hedged = dv01.copy()
    for t, h in zip(hedge_tenors, hedge_dv01):
        hedged[t] += h
    var_after, _ = pca_var(hedged.values, L, factor_var)
    print("Hedge DV01 by tenor (negative = sell futures/pay swaps):",
          {t: round(h) for t, h in zip(hedge_tenors, hedge_dv01)})
    print(f"PCA 99% 1-day VaR after hedge:  ${var_after:,.0f}")

    # 🧠 ML: backtest on unseen days using FULL tenor moves, not the PCA approximation
    pnl_before = -(test.values @ dv01.values)
    pnl_after = -(test.values @ hedged.values)
    print(f"\nOut-of-sample daily P&L std: before ${pnl_before.std():,.0f}  after ${pnl_after.std():,.0f}")
    breaches = (pnl_before < -var_before).mean()
    print(f"VaR breach rate (unhedged) on test: {breaches:.2%} (expect ~1%; higher = regime shift)")

    stab = stability(df)
    print("\nRolling-window stability (|cosine| vs full-sample loadings):")
    print(stab.round(3).to_string(index=False))
    weak = stab[stab[["PC1", "PC2", "PC3"]].min(axis=1) < 0.95]
    print(f"Windows with a factor shape below 0.95 similarity: {len(weak)} of {len(stab)}")

    # 🧠 ML: save the fitted pipeline so the risk engine applies the SAME factors every morning
    path = os.path.join(tempfile.mkdtemp(), "yield_pca.joblib")
    joblib.dump(pipe, path)
    print(f"\nSaved pipeline -> {os.path.basename(path)}")
    print("Model-risk note (SR 11-7): document factor choice, re-estimate window, VaR backtest")
    print("breaches and stability results; recalibrate when volatility regime changes.")

**Try this next:**
- Change the 1.8 volatility multiplier to 1.0 and see the VaR breach rate drop toward 1%.
- Fit the model on only the last 250 training days so it reflects recent volatility, and compare breaches.
- Hedge with only 10Y (k=1 style) and measure how much P&L volatility remains.